# Assignment 4 (new data) — 3. Intel scene images (CNN)

**One learning problem, three implementations: NumPy from scratch, TensorFlow/Keras, and PyTorch.**

| # | Dataset | Structure | Model family |
|---|---------|-----------|--------------|
| A | BRFSS diabetes | tabular, 19 features | MLP |
| B | Rice images | image, 3×32×32 | CNN |
| **C** | **Intel Image Classification** | **image, 3×64×64** | **CNN** |

Six classes of natural scene — `buildings`, `forest`, `glacier`, `mountain`, `sea`, `street` — photographed
in the wild. After the rice grains, this is the hard image problem: varied lighting, varied viewpoint,
cluttered content, and two genuinely ambiguous class pairs.

---

## ⚠️ This notebook cannot run on the data as delivered

`00_inventory.ipynb` establishes the reason. `data/` contains **only** `seg_pred/` — 7,301 loose `.jpg`
files with no class sub-folders. The labelled splits `seg_train/` and `seg_test/` are absent.

`seg_pred` is the original Analytics Vidhya competition's *prediction* set. It never had public labels, and
in this dataset the **folder name is the label** — so with no class folders there is no ground truth, and
therefore no way to train a classifier or to measure an accuracy.

**To complete the dataset**, from the repository root:

```bash
pip install kaggle          # needs a token at ~/.kaggle/kaggle.json
kaggle datasets download -d puneet6060/intel-image-classification -p data/ --unzip
```

That restores `data/seg_train/` (~14,000 labelled images) and `data/seg_test/` (~3,000). The loader accepts
both the flat layout and Kaggle's double-nested `seg_train/seg_train/<class>/` one, so no path editing is
needed afterwards.

Everything below is written and ready. Section 1 stops immediately with these instructions if the labelled
splits are still missing, rather than failing somewhere in the middle of a training run. Section 9 uses
`seg_pred` for the one thing it *is* good for — an unlabelled inference demo.

In [ ]:
import os, sys, json, platform

os.environ.setdefault("TF_CPP_MIN_LOG_LEVEL", "2")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

sys.path.insert(0, os.path.abspath("."))        # notebook/  -> ass4_newdata
sys.path.insert(0, os.path.abspath(".."))       # repo root  -> ass4_utils, scratch_nn

import ass4_newdata as D
import ass4_utils as U
import scratch_nn as S

U.set_seed(U.SEED)

import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader
import tensorflow as tf
import keras

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
# Outputs stay inside notebook/: weights in notebook/models/, metrics in
# notebook/results/. The six original notebooks keep theirs under the repo-root
# results/, so nothing here can overwrite them.
RESULTS = D.notebook_results_dir()
MODELS = D.use_notebook_model_dir()
print("weights ->", os.path.relpath(MODELS, D.ROOT))
print("metrics ->", os.path.relpath(RESULTS, D.ROOT))
plt.rcParams["figure.dpi"] = 110

print("torch     ", torch.__version__, "| device:", DEVICE,
      "|", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU only")
print("tensorflow", tf.__version__, "| GPUs:", tf.config.list_physical_devices("GPU") or "none (CPU)")

---
## 1. The data

The guard below is the first thing that runs. If it raises, stop here and restore the labelled splits with
the `kaggle` command above — nothing further in this notebook can produce a meaningful number without them.

In [ ]:
# Fail fast and with instructions, rather than halfway through training.
D.require_intel_labels()
print("labelled splits present - this notebook can run")

display(D.inventory(verbose=False))

Unlike the rice images, this dataset ships with the publisher's **own train/test split**, so
`load_intel_scene` uses it as given rather than inventing one. That is the more honest choice: it is the
split every published result on this dataset is measured against.

Images are ~150×150 but *not exactly* — `data/intel_image_classification_info.md` warns that a few are off
by a pixel or two, so the loader resizes everything to a uniform **64×64**. Scene recognition needs more
spatial detail than counting rice grains did, which is why this is 64×64 where notebook 02 was 32×32.

In [ ]:
IMG_SIZE = 64
SUBSET_TRAIN, SUBSET_TEST = 3_000, 1_500

X_train, y_train, X_test, y_test, meta = D.load_intel_scene(
    img_size=IMG_SIZE, n_train=SUBSET_TRAIN, n_test=SUBSET_TEST)

C, H, W = meta["shape"]
CLASSES = meta["classes"]
N_CLASSES = len(CLASSES)
DATASET_SUB = meta["name"]

print(f"\ntrain {X_train.shape}   test {X_test.shape}   dtype {X_train.dtype}")
print(f"classes: {CLASSES}")
print(f"\nper-class train counts: {dict(zip(CLASSES, np.bincount(y_train, minlength=N_CLASSES)))}")
print(f"per-class test  counts: {dict(zip(CLASSES, np.bincount(y_test, minlength=N_CLASSES)))}")

In [ ]:
U.plot_samples(X_train, y_train, CLASSES, n=12,
               title=f"{DATASET_SUB} - {IMG_SIZE}x{IMG_SIZE} inputs as the network sees them")
plt.show()

---
## 2. The shared architecture

64×64 inputs are four times the pixels of notebook 02, so a **third** convolutional block is added. Each
block still follows the same `Conv → ReLU → Pool` pattern from `theory_notes.md` §2.8, and each halves the
spatial size while doubling the channel count — the standard trade of resolution for semantic depth:

$$
X_{3\times64\times64}
\rightarrow \text{Conv}_{3\rightarrow32} \rightarrow 32\times32
\rightarrow \text{Conv}_{32\rightarrow64} \rightarrow 16\times16
\rightarrow \text{Conv}_{64\rightarrow128} \rightarrow 8\times8
\rightarrow \text{Flatten}_{8192} \rightarrow \text{Dense}_{128} \rightarrow \text{Dense}_{6}
$$

Why three blocks and not two: `theory_notes.md` §3.2(d) — each block learns one level of abstraction, and
the progression *edge → texture → shape → object part* needs depth to happen at all. Scenes are built from
object parts; rice grains are not.

In [ ]:
CH1, CH2, CH3, K, PAD, HIDDEN = 32, 64, 128, 3, 1, 128
EPOCHS, BATCH_SIZE, LR = 5, 128, 1e-3

H1, W1 = H // 2,  W // 2         # after block 1
H2, W2 = H1 // 2, W1 // 2        # after block 2
H3, W3 = H2 // 2, W2 // 2        # after block 3
FLAT = CH3 * H3 * W3

manual = [
    (f"Conv2D {C}->{CH1} 3x3",       (C * K * K) * CH1 + CH1,     f"{CH1}x{H}x{W}"),
    ("ReLU + MaxPool2D 2x2",         0,                           f"{CH1}x{H1}x{W1}"),
    (f"Conv2D {CH1}->{CH2} 3x3",     (CH1 * K * K) * CH2 + CH2,   f"{CH2}x{H1}x{W1}"),
    ("ReLU + MaxPool2D 2x2",         0,                           f"{CH2}x{H2}x{W2}"),
    (f"Conv2D {CH2}->{CH3} 3x3",     (CH2 * K * K) * CH3 + CH3,   f"{CH3}x{H2}x{W2}"),
    ("ReLU + MaxPool2D 2x2",         0,                           f"{CH3}x{H3}x{W3}"),
    ("Flatten",                      0,                           f"{FLAT}"),
    (f"Dense {FLAT}->{HIDDEN}",      FLAT * HIDDEN + HIDDEN,      f"{HIDDEN}"),
    ("ReLU",                         0,                           f"{HIDDEN}"),
    (f"Dense {HIDDEN}->{N_CLASSES}", HIDDEN * N_CLASSES + N_CLASSES, f"{N_CLASSES}"),
]
EXPECTED_PARAMS = sum(p for _, p, _ in manual)
ARCH = "SceneCNN 3conv+fc"

print(f"{'layer':<26}{'params':>12}   {'output shape'}")
print("-" * 60)
for name, p, shape in manual:
    print(f"{name:<26}{p:>12,}   {shape}")
print("-" * 60)
print(f"{'TOTAL (by hand)':<26}{EXPECTED_PARAMS:>12,}")

conv = sum(p for n, p, _ in manual if n.startswith("Conv"))
print(f"\nconvolutions {conv:,} ({conv / EXPECTED_PARAMS:.1%})   "
      f"classifier head {EXPECTED_PARAMS - conv:,} ({1 - conv / EXPECTED_PARAMS:.1%})")
print(f"\nshared hyperparameters: epochs={EPOCHS}, batch_size={BATCH_SIZE}, optimizer=Adam(lr={LR})")

results = []

---
## 3. Implementation A — from scratch (NumPy)

The from-scratch leg runs on the **3,000-image subset**. At 64×64 each `im2col` expansion is four times the
size of notebook 02's, so this is the slowest single cell in the whole assignment — expect minutes, not
seconds. That cost is the point: it is what the framework legs are hiding.

In [ ]:
U.set_seed(U.SEED)

def build_scratch_cnn():
    return S.Sequential([
        S.Conv2D(C, CH1, k=K, stride=1, pad=PAD, seed=1),
        S.ReLU(),
        S.MaxPool2D(2, 2),
        S.Conv2D(CH1, CH2, k=K, stride=1, pad=PAD, seed=2),
        S.ReLU(),
        S.MaxPool2D(2, 2),
        S.Conv2D(CH2, CH3, k=K, stride=1, pad=PAD, seed=3),
        S.ReLU(),
        S.MaxPool2D(2, 2),
        S.Flatten(),
        S.Dense(FLAT, HIDDEN, seed=4),
        S.ReLU(),
        S.Dense(HIDDEN, N_CLASSES, seed=5),
    ])

scratch_model = build_scratch_cnn()
print(scratch_model.summary(input_shape=(C, H, W)))
print(f"\nmatches hand count {EXPECTED_PARAMS:,}? {scratch_model.n_params() == EXPECTED_PARAMS}")

### Are the hand-written gradients actually right?

The convolution and pooling backward passes in `scratch_nn.py` were derived by hand, so they are checked
against central finite differences on a tiny model in float64 before any number from this leg is trusted.

**Why the step size is 1e-5 and not the default 1e-3.** `MaxPool2D` routes the gradient to the largest
element of each window, so the loss is only piecewise differentiable: where two elements are tied, an
epsilon-sized nudge can change which one wins, and the numerical derivative then measures a different
branch from the one the analytic gradient took. A coarse probe reports a large error on a backward pass
that is actually correct. Notebook 02 measures this directly on the rice images; the MLP in notebook 01
has no pooling layer and passes at the default step.

In [ ]:
U.set_seed(U.SEED)
check_model = S.Sequential([
    S.Conv2D(C, 4, k=3, stride=1, pad=1, seed=1),
    S.ReLU(),
    S.MaxPool2D(2, 2),
    S.Flatten(),
    S.Dense(4 * (H // 2) * (W // 2), N_CLASSES, seed=2),
])
# eps=1e-5, not the 1e-3 default: see the note above on max-pool ties.
err = S.gradient_check(check_model, X_train[:4], y_train[:4], n_samples=6, eps=1e-5)
print(f"worst relative error vs finite differences: {err:.3e}")
print("PASS - hand-derived gradients agree with the numerical derivative" if err < 1e-4
      else "FAIL - backward pass disagrees with finite differences")

In [ ]:
U.set_seed(U.SEED)
with U.Timer() as t_scratch:
    hist_scratch = S.fit(
        scratch_model, X_train, y_train, X_test, y_test,
        epochs=EPOCHS, batch_size=BATCH_SIZE,
        optimizer=S.Adam(LR), seed=U.SEED,
    )
print(f"\ntotal training time: {t_scratch.seconds:.1f}s")

In [ ]:
pred_scratch = scratch_model.predict_classes(X_test)
m = U.evaluate(y_test, pred_scratch, N_CLASSES)

results.append(U.RunResult(
    framework="Scratch (NumPy)", dataset=DATASET_SUB, model=ARCH,
    n_params=scratch_model.n_params(), epochs=EPOCHS,
    train_seconds=round(t_scratch.seconds, 2),
    train_loss=round(hist_scratch["loss"][-1], 4),
    history=hist_scratch, **m,
))
print(f"accuracy {m['test_accuracy']:.4f}   macro-F1 {m['f1_macro']:.4f}")

---
## 4. Implementation B — TensorFlow / Keras

Keras is channels-last, so the arrays are transposed — a layout change only.

> **Note on hardware.** Windows-native TensorFlow has no GPU build since 2.10, so this leg is on CPU while
> PyTorch uses the GPU where present. Parameter counts and accuracies compare; **wall-clock times do not.**

In [ ]:
U.set_seed(U.SEED)

Xtr_k = np.transpose(X_train, (0, 2, 3, 1))
Xte_k = np.transpose(X_test,  (0, 2, 3, 1))

def build_keras_cnn(name="scene_cnn"):
    m = keras.Sequential([
        keras.layers.Input(shape=(H, W, C)),
        keras.layers.Conv2D(CH1, K, padding="same", activation="relu"),
        keras.layers.MaxPooling2D(2),
        keras.layers.Conv2D(CH2, K, padding="same", activation="relu"),
        keras.layers.MaxPooling2D(2),
        keras.layers.Conv2D(CH3, K, padding="same", activation="relu"),
        keras.layers.MaxPooling2D(2),
        keras.layers.Flatten(),
        keras.layers.Dense(HIDDEN, activation="relu"),
        keras.layers.Dense(N_CLASSES),
    ], name=name)
    m.compile(optimizer=keras.optimizers.Adam(learning_rate=LR),
              loss=keras.losses.SparseCategoricalCrossentropy(from_logits=True),
              metrics=["accuracy"])
    return m

keras_model = build_keras_cnn()
keras_model.summary()
print(f"\nmatches hand count {EXPECTED_PARAMS:,}? {keras_model.count_params() == EXPECTED_PARAMS}")

In [ ]:
U.set_seed(U.SEED)
with U.Timer() as t_keras:
    hk = keras_model.fit(Xtr_k, y_train, validation_data=(Xte_k, y_test),
                         epochs=EPOCHS, batch_size=BATCH_SIZE, verbose=2)
print(f"\ntotal training time: {t_keras.seconds:.1f}s")

In [ ]:
pred_keras = keras_model.predict(Xte_k, batch_size=256, verbose=0).argmax(axis=1)
m = U.evaluate(y_test, pred_keras, N_CLASSES)

hist_keras = {"loss": [float(v) for v in hk.history["loss"]],
              "val_acc": [float(v) for v in hk.history["val_accuracy"]]}

results.append(U.RunResult(
    framework="TensorFlow/Keras", dataset=DATASET_SUB, model=ARCH,
    n_params=int(keras_model.count_params()), epochs=EPOCHS,
    train_seconds=round(t_keras.seconds, 2),
    train_loss=round(hist_keras["loss"][-1], 4),
    history=hist_keras, **m,
))
print(f"accuracy {m['test_accuracy']:.4f}   macro-F1 {m['f1_macro']:.4f}")

---
## 5. Implementation C — PyTorch

In [ ]:
U.set_seed(U.SEED)

class SceneCNN(nn.Module):
    def __init__(self, in_ch, n_classes):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(in_ch, CH1, K, padding=PAD), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(CH1, CH2, K, padding=PAD),   nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(CH2, CH3, K, padding=PAD),   nn.ReLU(), nn.MaxPool2d(2),
        )
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(FLAT, HIDDEN),
            nn.ReLU(),
            nn.Linear(HIDDEN, n_classes),
        )

    def forward(self, x):
        return self.classifier(self.features(x))

torch_model = SceneCNN(C, N_CLASSES).to(DEVICE)
n_torch = sum(p.numel() for p in torch_model.parameters())
print(torch_model)
print(f"\nparameters: {n_torch:,}")
print(f"matches hand count {EXPECTED_PARAMS:,}? {n_torch == EXPECTED_PARAMS}")

In [ ]:
def train_torch(model, Xtr, ytr, Xte, yte, epochs=EPOCHS, batch_size=BATCH_SIZE, lr=LR):
    # The explicit five-step loop, reused for the subset and the full-data run.
    loader = DataLoader(TensorDataset(torch.tensor(Xtr), torch.tensor(ytr)),
                        batch_size=batch_size, shuffle=True)
    Xte_t = torch.tensor(Xte)
    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    hist = {"loss": [], "val_acc": []}

    with U.Timer() as t:
        for epoch in range(1, epochs + 1):
            model.train()
            running, nb = 0.0, 0
            for xb, yb in loader:
                xb, yb = xb.to(DEVICE), yb.to(DEVICE)
                optimizer.zero_grad()          # 1. zero gradient
                logits = model(xb)             # 2. forward
                loss = criterion(logits, yb)   # 3. loss
                loss.backward()                # 4. backward (autograd)
                optimizer.step()               # 5. update
                running += loss.item(); nb += 1

            model.eval()
            preds = []
            with torch.no_grad():
                for i in range(0, len(Xte_t), 512):
                    preds.append(model(Xte_t[i:i + 512].to(DEVICE)).argmax(1).cpu())
            acc = float((torch.cat(preds).numpy() == yte).mean())

            hist["loss"].append(running / nb)
            hist["val_acc"].append(acc)
            print(f"epoch {epoch}/{epochs}  loss {running/nb:.4f}  test_acc {acc:.4f}")

    return hist, t.seconds


def predict_torch(model, X):
    model.eval()
    out = []
    with torch.no_grad():
        for i in range(0, len(X), 512):
            out.append(model(torch.tensor(X[i:i + 512]).to(DEVICE)).argmax(1).cpu())
    return torch.cat(out).numpy()


U.set_seed(U.SEED)
hist_torch, secs_torch = train_torch(torch_model, X_train, y_train, X_test, y_test)
print(f"\ntotal training time: {secs_torch:.1f}s")

In [ ]:
pred_torch = predict_torch(torch_model, X_test)
m = U.evaluate(y_test, pred_torch, N_CLASSES)

results.append(U.RunResult(
    framework="PyTorch", dataset=DATASET_SUB, model=ARCH,
    n_params=n_torch, epochs=EPOCHS,
    train_seconds=round(secs_torch, 2),
    train_loss=round(hist_torch["loss"][-1], 4),
    history=hist_torch, **m,
))
print(f"accuracy {m['test_accuracy']:.4f}   macro-F1 {m['f1_macro']:.4f}")

---
## 6. Scaling up — the full labelled split

Same architecture, all ~14,000 training images, Keras and PyTorch only.

In [ ]:
X_tr_f, y_tr_f, X_te_f, y_te_f, meta_f = D.load_intel_scene(img_size=IMG_SIZE)
DATASET_FULL = meta_f["name"]
print(f"\ntrain {X_tr_f.shape}   test {X_te_f.shape}")
print(f"memory: train {X_tr_f.nbytes / 1e9:.2f} GB, test {X_te_f.nbytes / 1e9:.2f} GB")

In [ ]:
U.set_seed(U.SEED)
torch_full = SceneCNN(C, N_CLASSES).to(DEVICE)
hist_tf, secs_tf = train_torch(torch_full, X_tr_f, y_tr_f, X_te_f, y_te_f)

pred = predict_torch(torch_full, X_te_f)
m = U.evaluate(y_te_f, pred, N_CLASSES)
results.append(U.RunResult(
    framework="PyTorch", dataset=DATASET_FULL, model=ARCH,
    n_params=sum(p.numel() for p in torch_full.parameters()), epochs=EPOCHS,
    train_seconds=round(secs_tf, 2), train_loss=round(hist_tf["loss"][-1], 4),
    history=hist_tf, **m,
))
print(f"\naccuracy {m['test_accuracy']:.4f}   macro-F1 {m['f1_macro']:.4f}")

In [ ]:
U.set_seed(U.SEED)
keras_full = build_keras_cnn("scene_cnn_full")
Xtr_kf = np.transpose(X_tr_f, (0, 2, 3, 1))
Xte_kf = np.transpose(X_te_f, (0, 2, 3, 1))

with U.Timer() as t_kf:
    hkf = keras_full.fit(Xtr_kf, y_tr_f, validation_data=(Xte_kf, y_te_f),
                         epochs=EPOCHS, batch_size=BATCH_SIZE, verbose=2)

pred = keras_full.predict(Xte_kf, batch_size=256, verbose=0).argmax(axis=1)
m = U.evaluate(y_te_f, pred, N_CLASSES)
hist_kf = {"loss": [float(v) for v in hkf.history["loss"]],
           "val_acc": [float(v) for v in hkf.history["val_accuracy"]]}
results.append(U.RunResult(
    framework="TensorFlow/Keras", dataset=DATASET_FULL, model=ARCH,
    n_params=int(keras_full.count_params()), epochs=EPOCHS,
    train_seconds=round(t_kf.seconds, 2), train_loss=round(hist_kf["loss"][-1], 4),
    history=hist_kf, **m,
))
print(f"\naccuracy {m['test_accuracy']:.4f}   macro-F1 {m['f1_macro']:.4f}")

---
## 7. Comparison

In [ ]:
table = U.results_table(results)
display(table)
print(f"\nall parameter counts identical: {table['n_params'].nunique() == 1} "
      f"({table['n_params'].iloc[0]:,})")

In [ ]:
U.plot_history({f"{r.framework} [{r.dataset}]": r.history for r in results},
               title="Intel scenes - one CNN, three implementations")
plt.show()

In [ ]:
best = max(results, key=lambda r: r.test_accuracy)
fig, ax = plt.subplots(figsize=(6.2, 5.4))
U.plot_confusion(best.confusion, CLASSES,
                 title=f"{best.framework} on {best.dataset} (acc {best.test_accuracy:.4f})", ax=ax)
fig.tight_layout(); plt.show()

### 7.1 The two confusions the dataset is known for

`data/intel_image_classification_info.md` predicts two failure modes before any model is trained:
**glacier ↔ mountain** (a snow-covered mountain is both) and **buildings ↔ street** (a street photograph
almost always contains buildings). The cell below checks whether the model actually made those mistakes,
rather than taking the claim on trust.

In [ ]:
cm = np.array(best.confusion, dtype=float)
off = cm.copy()
np.fill_diagonal(off, 0)

pairs = [(CLASSES[i], CLASSES[j], int(off[i, j]))
         for i in range(N_CLASSES) for j in range(N_CLASSES) if off[i, j] > 0]
pairs.sort(key=lambda t: -t[2])

print("most frequent confusions:\n")
for true, pred_c, n in pairs[:8]:
    print(f"  {true:<12} mistaken for {pred_c:<12} {n:>5} times")

expected = [("glacier", "mountain"), ("mountain", "glacier"),
            ("buildings", "street"), ("street", "buildings")]
rank = {(t, p): i for i, (t, p, _) in enumerate(pairs)}
print("\npredicted-by-the-info-file pairs, and where they actually rank:")
for pair in expected:
    r = rank.get(pair)
    print(f"  {pair[0]:<10} -> {pair[1]:<10} "
          + (f"rank #{r + 1} of {len(pairs)}" if r is not None else "did not occur"))

---
## 8. Where each component lives

In [ ]:
component_table = pd.DataFrame([
    ["Data loading",     "ass4_newdata -> NumPy arrays", "same arrays (transposed to NHWC)", "same arrays"],
    ["Train/test split", "publisher's own split",        "same",                       "same"],
    ["Tensor layout",    "N,C,H,W",                      "N,H,W,C (channels-last)",    "N,C,H,W"],
    ["Model definition", "S.Sequential([...])",          "keras.Sequential([...])",    "nn.Module"],
    ["Convolution",      "S.Conv2D (im2col by hand)",    "keras.layers.Conv2D",        "nn.Conv2d"],
    ["Pooling",          "S.MaxPool2D (argmax routing)", "keras.layers.MaxPooling2D",  "nn.MaxPool2d"],
    ["Activation",       "S.ReLU (mask multiply)",       "activation='relu'",          "nn.ReLU"],
    ["Loss",             "S.SoftmaxCrossEntropy",        "SparseCategoricalCrossentropy", "nn.CrossEntropyLoss"],
    ["Gradient",         "hand-derived backward()",      "automatic (GradientTape)",   "automatic (autograd)"],
    ["Optimizer",        "S.Adam (written out)",         "keras.optimizers.Adam",      "torch.optim.Adam"],
    ["Training loop",    "explicit for-loop",            "model.fit()",                "explicit for-loop"],
], columns=["Component", "Scratch", "TensorFlow/Keras", "PyTorch"])

display(component_table)

---
## 9. `seg_pred` — inference without ground truth

The 7,301 images in `seg_pred/` have no labels, so they cannot contribute a single number to any table
above. What they *can* do is demonstrate the trained model running on genuinely unseen data, which is the
form a live demo takes.

Read the output carefully: these are **predictions, not scores**. Nothing here can be called right or
wrong, because there is nothing to check against. The confidence value is the model's softmax output —
the network's own opinion, which says nothing about whether that opinion is correct.

In [ ]:
X_pred, pred_files = D.load_intel_unlabeled(img_size=IMG_SIZE, n=12)

# seg_pred is raw [0,1]; apply the SAME channel statistics the training split
# defined, otherwise the model sees a differently-scaled input.
mu = np.array(meta_f["channel_mean"], dtype=np.float32).reshape(1, 3, 1, 1)
sd = np.array(meta_f["channel_std"], dtype=np.float32).reshape(1, 3, 1, 1)
X_pred_n = (X_pred - mu) / sd

torch_full.eval()
with torch.no_grad():
    logits = torch_full(torch.tensor(X_pred_n).to(DEVICE))
    prob = torch.softmax(logits, dim=1).cpu().numpy()
guess = prob.argmax(1)

fig, axes = plt.subplots(2, 6, figsize=(15, 5.4))
for ax, img, g, p, fn in zip(axes.ravel(), X_pred, guess, prob, pred_files):
    ax.imshow(img.transpose(1, 2, 0))
    ax.set_title(f"{CLASSES[g]}\n{p[g]:.0%} conf", fontsize=8)
    ax.axis("off")
fig.suptitle("seg_pred - model predictions on unlabelled images (NOT accuracy: there is no ground truth)")
fig.tight_layout(); plt.show()

print("mean confidence of the top class:", f"{prob.max(axis=1).mean():.3f}")
print("predicted class distribution:",
      dict(zip(CLASSES, np.bincount(guess, minlength=N_CLASSES))))

In [ ]:
payload = {
    "meta": {k: v for k, v in meta.items() if k != "feature_names"},
    "subset": {"n_train": int(SUBSET_TRAIN), "n_test": int(SUBSET_TEST),
               "img_size": int(IMG_SIZE)},
    "results": [r.__dict__ for r in results],
    "component_table": component_table.to_dict(orient="records"),
    "seg_pred_note": ("7,301 unlabelled images; used for an inference demo only, "
                      "never for training or scoring"),
}
path = os.path.join(RESULTS, "n03_intel_scene_cnn.json")
with open(path, "w", encoding="utf-8") as f:
    json.dump(payload, f, indent=2, default=str)
print("saved", os.path.relpath(path, D.HERE))

In [ ]:
TAG = "n03_intel_scene_cnn"
by_key = {(r.framework, r.dataset): r for r in results}

for obj, fw, ds, stem in [
    (scratch_model, "Scratch (NumPy)",  DATASET_SUB,  "scene_cnn_scratch_subset"),
    (keras_model,   "TensorFlow/Keras", DATASET_SUB,  "scene_cnn_keras_subset"),
    (torch_model,   "PyTorch",          DATASET_SUB,  "scene_cnn_torch_subset"),
    (keras_full,    "TensorFlow/Keras", DATASET_FULL, "scene_cnn_keras_full"),
    (torch_full,    "PyTorch",          DATASET_FULL, "scene_cnn_torch_full"),
]:
    r = by_key[(fw, ds)]
    U.save_model(obj, stem, TAG, architecture=r.model, dataset=r.dataset,
                 epochs=r.epochs, test_accuracy=r.test_accuracy, f1_macro=r.f1_macro,
                 img_size=IMG_SIZE, classes=CLASSES)

display(U.list_models(TAG)[["name", "framework", "n_params", "dataset", "test_accuracy"]])

---
## 10. What this dataset shows

**An incomplete download is a result too.** The single most useful thing this notebook did was refuse to
run. `seg_pred` looks like 7,301 perfectly good images, and a loader that globbed for `*.jpg` without
checking for class folders would have produced *something* — a silently mislabelled dataset, or an
accuracy figure computed against nonsense. Checking the structure before trusting it is what
`00_inventory.ipynb` exists for.

**This is the hard image problem of the three.** Rice grains are pre-segmented, centred, and perfectly
balanced; natural scenes are none of those. Expect noticeably lower accuracy than notebook 02, and expect
the errors to cluster in the `glacier`/`mountain` and `buildings`/`street` pairs that §7.1 checks for —
those are genuine category overlaps in the world, not defects in any of the three implementations.

**The three implementations still agree.** Third dataset, third confirmation: identical parameter counts,
accuracies within noise. What changes between the legs is visibility and cost, not the function being
learned.

**Confidence is not correctness.** Section 9 ends on the distinction that matters most for the demo: a
softmax output near 1.0 means the network is confident, and on unlabelled data that is all it means.